# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m03_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 03. 비행 경로 이탈도 및 호버링 안정성 계산 1차 숙련도 채점 Baseline 알고리즘 구현
**소요시간**: 약 2.5시간 | **필수 산출물**: src/models/trn01_baseline_model.py (필수), 기본 추론 결과 보고서 (필수)

### 📋 미션 개요 (Overview)
정제된 비행로그를 바탕으로 조종자의 기본 비행 역량을 정량화하는 기준선(Baseline) 채점 체계를 구축한다. 경로 이탈도(RMSE), 호버링 안정성, 고도 유지 오차를 비행 조건별로 표준화하고 가중치를 1차 튜닝하여 0~100점 및 A/B/C 등급을 산출하는 모델(`trn01_baseline_model.py`)을 개발한다. 교관 평가와의 등급 일치율(75% 이상)을 확인하고, 혼동표 분석을 통해 돌발 상황 대응 미반영 등 Baseline의 한계를 도출한다.

### 🎓 학습 목표 (Objectives)
도메인 지표를 결합해 **해석 가능한 기준선(Baseline) 채점 모델을 설계하고, 조건 표준화 효과와 모델 한계를 진단하는 평가 역량**을 익힌다. 경로 RMSE, 호버링 이탈·흔들림, 고도 오차를 환경 조건별로 표준화해 0~100점·A/B/C 등급으로 매핑하는 가중치 튜닝 과정을 실습하고, 교관 등급과의 일치율 및 혼동표 분석을 통해 Baseline이 돌발 상황 대응을 놓치는 구조적 원인을 규명하는 방법을 배운다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션3 코드를 실행해 채점식 가중치와 등급 일치율을 확인한다.
#### 이 단계의 역할
미션3 코드를 실행해 채점식 가중치와 등급 일치율을 확인한다.

#### 해야 할 일

- 미션3 코드를 실행해 채점식 가중치와 등급 일치율을 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 등급 일치율 75% 이상(예시 75.8%), 점수 평균 오차 5.3점 가장 많은 오류: 교관 C -> Baseline B(돌발 대응 미반영)

#### 학습자가 제출해야 하는 내용

- 미션3 코드를 실행해 채점식 가중치와 등급 일치율을 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 경로 RMSE, 호버링 이탈·흔들림, 고도 오차를 0~100점으로 환산해 A/B/C 등급을 부여하는 Baseline 채점 함수를 작성하고, 교관 등급과의 일치율 및 평균 점수 오차를 계산해줘.
**AI 응답 검증 포인트**

- 등급 일치율(75% 이상)과 평균 점수 오차가 실제 평가셋에서 재현되는지 확인
- 가중치 합이 1.0이며 점수 클리핑(0~100) 경계 조건에 오류가 없는지 점검

#### Step 2. 조건 표준화 유무에 따른 조건별 정확도를 비교한다.
#### 이 단계의 역할
조건 표준화 유무에 따른 조건별 정확도를 비교한다.

#### 해야 할 일

- 조건 표준화 유무에 따른 조건별 정확도를 비교한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 등급 일치율 75% 이상(예시 75.8%), 점수 평균 오차 5.3점 가장 많은 오류: 교관 C -> Baseline B(돌발 대응 미반영)

#### 학습자가 제출해야 하는 내용

- 조건 표준화 유무에 따른 조건별 정확도를 비교한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 비행 조건(주간/야간/돌풍)별 난이도 차이를 반영해 지표를 Z-score 또는 조건별 기준으로 표준화했을 때와 미적용 시의 등급 정확도 차이를 비교 분석해줘.
**AI 응답 검증 포인트**

- 돌풍·야간 조건에서 과도하게 감점되던 조종자의 등급 편향이 표준화 후 완화됐는지 확인
- 표준화 기준 통계량(평균/표준편차)이 평가 데이터가 아닌 기준 집단에서만 계산됐는지(데이터 누수 방지) 점검

#### Step 3. 혼동표에서 가장 많은 오류 유형과 그 원인을 정리한다.
#### 이 단계의 역할
혼동표에서 가장 많은 오류 유형과 그 원인을 정리한다.

#### 해야 할 일

- 혼동표에서 가장 많은 오류 유형과 그 원인을 정리한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 등급 일치율 75% 이상(예시 75.8%), 점수 평균 오차 5.3점 가장 많은 오류: 교관 C -> Baseline B(돌발 대응 미반영)

#### 학습자가 제출해야 하는 내용

- 혼동표에서 가장 많은 오류 유형과 그 원인을 정리한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 교관 등급과 Baseline 예측 등급의 혼동행렬(Confusion Matrix)을 생성하고, 특히 '교관 C -> Baseline B'처럼 과대평가된 사례들의 공통 특징을 요약해줘.
**AI 응답 검증 포인트**

- 혼동행렬의 전체 합계가 평가 대상 비행 건수와 일치하는지 확인
- 오분류 사례가 실제 돌발 장애물 회피나 비상 착륙 구간 실패 때문인지 원본 로그로 검증

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
BASE_F에서 alt_err를 빼고 다시 실행해 일치율 변화를 보고, 고도 유지 오차가 숙련도를 얼마나 설명하는지 토의한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> Baseline 채점식의 가중치와 등급 경계값(cut-off)을 조정했을 때 일치율과 재교육 누락률(교관 C를 B로 오분류)이 어떻게 변하는지 민감도 분석 코드를 작성해줘.
**AI 응답 검증 포인트**

- 전체 일치율만 높이고 정작 위험군(C등급)을 놓치는 방향의 튜닝은 아닌지 확인
- 변경된 가중치의 군사적·훈련적 해석이 타당한지 근거를 검토


In [ ]:
"""비행 경로 이탈도(경로 RMSE)와 호버링 안정성(평균 이탈·흔들림), 고도 유지 오차로 1차 숙련도 점수(0~100)와 등급(A/B/C)을 계산하는 Baseline 채점 알고리즘을 구현하고, 교관 평가 등급과의 일치율(처리 정확도)과 판단 지연을 측정한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    # [TODO : 학생 실습 3] 비행 경로 이탈도 및 호버링 안정성 계산 1차 숙련도 채점 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    # [TODO : 학생 실습 3] 비행 경로 이탈도 및 호버링 안정성 계산 1차 숙련도 채점 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

# ---- 비행로그 특징 추출 공통 함수(여러 조종자 로그를 한 번에 계산) ----
from scipy.ndimage import uniform_filter1d

BASE_F = ["path_rmse", "hover_drift", "hover_std", "alt_err"]
ADV_F = BASE_F + ["obs_rt", "obs_lateral", "touch_vz", "att_exceed", "jerk", "overshoot", "n_risk"]
RULES = {"near_collision": ("obs_lateral", "<", 2.5), "hard_landing": ("touch_vz", ">", 2.0), "attitude_exceed": ("att_exceed", ">=", 10)}


def detect(df):
    """위험 조작 사건 탐지 규칙: 장애물 2.5초 후 회피 거리 2.5m 미만, 접지 속도 2.0m/s 초과, 자세 35도 초과 1초 이상."""
    out = {}
    for ev, (f, op, th) in RULES.items():
        out[ev] = df[f] < th if op == "<" else (df[f] > th if op == ">" else df[f] >= th)
    return pd.DataFrame(out)


def features(log):
    """경로 이탈도·호버링 안정성(기본) + 장애물 반응·착륙 속도·자세 한계·급조작·과보정(고도화) 특징."""
    X, Y, A = (uniform_filter1d(log[k], 9, axis=0) for k in ("x", "y", "alt"))       # GPS 잡음 완화(이동평균 0.9초)
    h = np.hypot(X - log["tx"], Y - log["ty"])
    cr, hv = slice(0, 600), slice(620, 850)
    k0, k1 = int(OBST_T / DT), int((OBST_T + 2.5) / DT)
    side = np.sign(log["ty"][k0 + 1] - log["ty"][k0 - 1])
    moved = (log["stick_roll"][k0:k0 + 40] * side) > 0.3
    rt = np.where(moved.any(0), moved.argmax(0) * DT, 4.0)
    vz = -np.gradient(A, DT, axis=0)
    land = np.argmax((A < 0.4) & (T[:, None] >= LAND_T), axis=0)
    touch = np.array([vz[max(i - 6, 0):i - 2, j].mean() if i > 6 else 3.0 for j, i in enumerate(land)])
    ds = np.abs(np.diff(np.stack([log["stick_roll"], log["stick_pitch"]]), axis=1)) / DT
    return pd.DataFrame({
        "path_rmse": np.sqrt((h[cr] ** 2).mean(0)), "hover_drift": h[hv].mean(0), "hover_std": np.hypot(X[hv].std(0), Y[hv].std(0)),
        "alt_err": np.abs(A[:850] - log["talt"][:850]).mean(0),
        "obs_rt": rt, "obs_lateral": np.abs(Y[k1] - Y[k0]), "touch_vz": touch,
        "att_exceed": (np.maximum(np.abs(log["roll_deg"]), np.abs(log["pitch_deg"])) > 35).sum(0),
        "jerk": np.percentile(ds, 95, axis=(0, 1)),
        "overshoot": np.clip(np.abs(Y[k0:k0 + 120] - Y[k0]).max(0) - 6.0, 0, None)}).pipe(
        lambda d: d.assign(n_risk=detect(d).sum(axis=1)))                                     # overshoot: 회피 후 목표 6m를 넘어선 거리(과보정)


def make_dataset(n_pilots=40, seed=7, conds=("day", "night", "gust"), flights=1):
    """조종자 n명 x 조건 x 반복 비행 -> 비행별 특징 + 정답(교관 점수·등급·위험 사건) 표."""
    pilots = make_pilots(n_pilots, seed)
    out = []
    for c in conds:
        for f in range(flights):
            log, tr = fly(pilots, c, seed=seed * 100 + list(COND).index(c) * 10 + f)
            out.append(pd.concat([tr.reset_index(drop=True), features(log)], axis=1).assign(flight=f))
    return pd.concat(out, ignore_index=True)


def cond_z(df, ref, cols):
    """조건별 기준 집단(ref)의 평균·표준편차로 표준화: 돌풍처럼 어려운 조건에서 받는 불이익을 줄인다."""
    z = df.copy()
    z[cols] = z[cols].astype(float)
    for c in df["cond"].unique():
        r = ref[ref["cond"] == c][cols]
        z.loc[df["cond"] == c, cols] = (df.loc[df["cond"] == c, cols] - r.mean()) / (r.std() + 1e-9)
    return z


def to_grade(score):
    return np.where(score >= 75, "A", np.where(score >= 55, "B", "C"))

# [교육용가상자료] 기준(튜닝)용 조종자 40명, 평가용 조종자 40명 x 조건 3종(주간·야간·돌풍)
train = make_dataset(40, seed=7)
test = make_dataset(40, seed=8)

# 1) 휴리스틱 가중치 1차 튜닝: 조건별 표준화 특징 -> 교관 점수 선형 근사
ztr, zte = cond_z(train, train, BASE_F), cond_z(test, train, BASE_F)
lr = LinearRegression().fit(ztr[BASE_F], train["instructor"])
print("채점식: 점수 = {:.1f}".format(lr.intercept_) + "".join(f" {w:+.1f}x{f}" for f, w in zip(BASE_F, lr.coef_)) + "  (특징은 조건별 표준화 값)")

t0 = time.perf_counter()
test["base_score"] = np.clip(lr.predict(zte[BASE_F]), 0, 100)
test["base_grade"] = to_grade(test["base_score"])
ms = (time.perf_counter() - t0) * 1000 / len(test)
acc = (test["base_grade"] == test["grade"]).mean()
print(f"평가 비행 {len(test)}회: 등급 일치율(처리 정확도) {acc:.1%} (목표 75% 이상: {'달성' if acc >= 0.75 else '미달'}), "
      f"점수 평균 오차 {np.abs(test['base_score'] - test['instructor']).mean():.1f}점, 채점 지연 {ms:.3f} ms/비행")

# 2) 조건별 정확도 + 조건 표준화를 하지 않았을 때 비교
raw = LinearRegression().fit(train[BASE_F], train["instructor"])
test["raw_grade"] = to_grade(raw.predict(test[BASE_F]))
print("\n조건    표준화 O   표준화 X")
for c in COND:
    s = test[test["cond"] == c]
    print(f"{c:6s}  {(s['base_grade'] == s['grade']).mean():8.1%}  {(s['raw_grade'] == s['grade']).mean():8.1%}")
print(f"전체    {acc:8.1%}  {(test['raw_grade'] == test['grade']).mean():8.1%}")
gust_raw = raw.predict(test[test["cond"] == "gust"][BASE_F]).mean() - raw.predict(test[test["cond"] == "day"][BASE_F]).mean()
print(f"-> 표준화하지 않으면 돌풍 비행 점수가 주간보다 평균 {gust_raw:+.1f}점 달라진다(조건 효과가 점수에 섞임). 표준화는 같은 조건 동료 대비 비교다.")

# 3) 혼동표와 한계 분석
print("\n[교관 등급 x Baseline 등급]")
print(pd.crosstab(test["grade"].rename("교관"), test["base_grade"].rename("Baseline")).to_string())
over = test[(test["grade"] == "C") & (test["base_grade"] != "C")]
ev = ["near_collision", "hard_landing"]
print(f"\n가장 많은 오류: 교관 C -> Baseline B {len(over)}건. 교관 C 비행의 {test[test['grade'] == 'C'][ev].any(axis=1).mean():.0%}가"
      f" 장애물 근접·경착륙을 포함하지만 Baseline 특징(경로·호버링·고도)에는 이 정보가 없다.")
print("-> 경로·호버링만 보는 Baseline은 돌발 상황 대응(장애물 회피, 비상 착륙)을 반영하지 못한다. 미션 4에서 반응 특징을 추가한다.")

plt.figure(figsize=(6, 5))
for c, col in zip(COND, ["tab:blue", "tab:purple", "tab:orange"]):
    s = test[test["cond"] == c]
    plt.scatter(s["instructor"], s["base_score"], s=15, c=col, label=c)
plt.plot([20, 100], [20, 100], "k--"); plt.axhline(75, c="gray", lw=0.5); plt.axhline(55, c="gray", lw=0.5)
plt.xlabel("instructor score"); plt.ylabel("baseline AI score"); plt.legend(); plt.title("Baseline skill score vs instructor (synthetic)")
plt.tight_layout(); plt.show()

# TODO: Baseline 모델을 src/models/trn01_baseline_model.py 로, 추론 결과 보고서를 docs/trn01_m03_baseline_report.md 로 저장하세요.


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
채점식: 점수 = 59.3 -0.4xpath_rmse -0.8xhover_drift +0.2xhover_std -13.0xalt_err  (특징은 조건별 표준화 값)
평가 비행 120회: 등급 일치율(처리 정확도) 75.8% (목표 75% 이상: 달성), 점수 평균 오차 5.3점, 채점 지연 0.025 ms/비행

조건    표준화 O   표준화 X
day        70.0%     75.0%
night      80.0%     70.0%
gust       77.5%     82.5%
전체       75.8%     75.8%
-> 표준화하지 않으면 돌풍 비행 점수가 주간보다 평균 -4.0점 달라진다(조건 효과가 점수에 섞임). 표준화는 같은 조건 동료 대비 비교다.

[교관 등급 x Baseline 등급]
Baseline   A   B   C
교관                  
A         15   6   0
B          3  26   4
C          0  16  50

가장 많은 오류: 교관 C -> Baseline B 16건. 교관 C 비행의 95%가 장애물 근접·경착륙을 포함하지만 Baseline 특징(경로·호버링·고도)에는 이 정보가 없다.
-> 경로·호버링만 보는 Baseline은 돌발 상황 대응(장애물 회피, 비상 착륙)을 반영하지 못한다. 미션 4에서 반응 특징을 추가한다.
```
</details>
